# Sentinel AI — Capstone Evidence Notebook

**Author:** Majed Mohamed Alsehli (ماجد محمد السهلي)  
**Training programme:** SDAIA Academy — Building Agentic AI Systems  
**Instructor:** محمد البلادي  
**Cohort:** 17 August 2025 – 21 May 2026  
**Declared track:** A — Supervisor + workers

This notebook is the executable evidence for all eight rubric sections. Restart the kernel, run every cell from top to bottom, resolve any failure honestly, and save all outputs before submission.

## 1. Reproducible setup and credential gate

The first cell installs the declared environment and fails early unless OpenAI and LangSmith are configured. Optional reputation-provider keys are reported only as booleans; no secret value is printed.

In [1]:
%pip install -q -r ../requirements.txt

import json
import os
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
load_dotenv('../.env', override=True)
sys.path.insert(0, '../src')

from sentinel.config import PROJECT_ROOT, langsmith_status
status = langsmith_status()
credential_status = {
    'OPENAI_API_KEY': bool(os.getenv('OPENAI_API_KEY')),
    'LANGCHAIN_API_KEY': bool(os.getenv('LANGCHAIN_API_KEY') or os.getenv('LANGSMITH_API_KEY')),
    'ABUSEIPDB_API_KEY_optional': bool(os.getenv('ABUSEIPDB_API_KEY')),
    'VIRUSTOTAL_API_KEY_optional': bool(os.getenv('VIRUSTOTAL_API_KEY')),
}
print('Python:', sys.version.split()[0])
print('Credential readiness:', credential_status)
print('LangSmith readiness:', status)
assert credential_status['OPENAI_API_KEY'], 'Configure OPENAI_API_KEY in ../.env'
assert status['tracing_enabled'], 'Set LANGCHAIN_TRACING_V2=true in ../.env'
assert status['api_key_configured'], 'Configure LANGCHAIN_API_KEY in ../.env'
print('Setup assertion: PASS')


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: /private/tmp/sentinel-audit.p698Ca/venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Python: 3.13.2
Credential readiness: {'OPENAI_API_KEY': True, 'LANGCHAIN_API_KEY': True, 'ABUSEIPDB_API_KEY_optional': False, 'VIRUSTOTAL_API_KEY_optional': False}
LangSmith readiness: {'tracing_enabled': True, 'api_key_configured': True, 'project': 'sentinel-ai-capstone'}
Setup assertion: PASS


## 2. Agent fundamentals — real tool execution and structured output

The email specialist is given one argument-dependent parsing tool. The LLM must choose it, Sentinel executes the call, returns a `ToolMessage`, and obtains a Pydantic `SpecialistAssessment`. This proves execution rather than merely showing a bound tool schema.

In [2]:
from sentinel.agents.specialists import run_email_agent

raw_email = '''From: Account Security <alerts@contoso-security.example>
Reply-To: recovery-team@example.net
Subject: Urgent — verify your account

Your password expires today. Verify your account at https://example.net/login immediately.'''
specialist_demo = run_email_agent(raw_email)
print('Structured result type:', type(specialist_demo).__name__)
print('Specialist assessment:', specialist_demo.assessment.model_dump())
print('Executed tool evidence:')
for item in specialist_demo.tool_evidence:
    print(json.dumps(item.model_dump(), indent=2, default=str))
assert specialist_demo.tool_evidence, 'The model did not choose a tool'
assert specialist_demo.tool_evidence[0].tool_name == 'extract_email_indicators'
assert specialist_demo.tool_evidence[0].output['urls'] == ['https://example.net/login']
print('Model-selected real tool execution assertion: PASS')

Structured result type: SpecialistResult
Specialist assessment: {'summary': 'The email exhibits several characteristics commonly associated with phishing attempts, including urgency and requests for immediate action regarding account verification.', 'notable_indicators': ['From: Account Security <alerts@contoso-security.example>', 'Reply-To: recovery-team@example.net', 'Subject: Urgent — verify your account', 'URLs: [https://example.net/login]', "Suspicious Phrases: 'urgent', 'verify your account', 'password expires'"], 'limitations': ["The analysis is based solely on the extracted indicators and does not include verification of the actual sender's legitimacy.", "The URL's safety has not been assessed; further investigation is required to determine if it leads to a legitimate site.", 'The context of the email (e.g., previous communications, account status) is not considered, which may affect the interpretation of the urgency.', "The tool's output is dependent on the accuracy of the par

## 3. Multi-agent routing — structured LLM supervisor

Sentinel uses the supervisor routing pattern because each dominant artifact requires a narrow specialist and tool set. The decision is made by `with_structured_output(RouteDecision)`; Python only dispatches the returned destination and contains no keyword classifier.

In [3]:
from sentinel.agents.supervisor import route_request
from sentinel.models.schemas import RouteDecision

routing_examples = [
    'Investigate network traffic from 1.1.1.1',
    'Review this raw email for credential phishing',
    'Check SHA-256 ' + 'a' * 64,
    'Assess https://example.com/login without opening it',
]
route_results = [route_request(question) for question in routing_examples]
for question, decision in zip(routing_examples, route_results):
    print(question, '=>', decision.model_dump())
assert all(isinstance(decision, RouteDecision) for decision in route_results)
assert {decision.destination for decision in route_results} == {
    'ip_agent', 'email_agent', 'file_agent', 'url_agent'
}
print('Structured LLM routing assertion: PASS')

Investigate network traffic from 1.1.1.1 => {'destination': 'ip_agent', 'reason': 'The investigation focuses on network traffic from the specific IP address 1.1.1.1, making it necessary to engage the IP specialist to analyze the network indicators associated with this address.'}
Review this raw email for credential phishing => {'destination': 'email_agent', 'reason': 'The investigation focuses on a raw email, which includes headers and message body, making it an email artifact. The primary object of the investigation is the content of the email itself, specifically for signs of credential phishing.'}
Check SHA-256 aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa => {'destination': 'file_agent', 'reason': 'The investigation is focused on a SHA-256 hash, which is a file artifact. Therefore, the file_agent is the appropriate specialist to handle this investigation.'}
Assess https://example.com/login without opening it => {'destination': 'url_agent', 'reason': "The investig

## 4. Hybrid RAG — load, split, embed, store, retrieve

**Hybrid RAG** is the explicit design choice. The local corpus supplies stable defensive procedure while live tools supply current reputation evidence. Two-step RAG alone would lack live observations; unconstrained agentic retrieval alone would make stable guidance less reproducible.

In [4]:
from sentinel.rag.loader import load_markdown_documents
from sentinel.rag.retriever import build_retriever
from sentinel.workflows.workflow import set_retriever

documents = load_markdown_documents()
retriever, chunks = build_retriever(documents, PROJECT_ROOT / 'chroma')
set_retriever(retriever)
print('Loaded documents:', len(documents))
print('Embedded/stored chunks:', len(chunks))
print('Sources:', [document.metadata['filename'] for document in documents])
assert len(documents) == 5 and len(chunks) >= 5
print('Load/split/embed/store assertion: PASS')

Loaded documents: 5
Embedded/stored chunks: 10
Sources: ['incident_response.md', 'ip_reputation.md', 'malware.md', 'phishing.md', 'url_analysis.md']
Load/split/embed/store assertion: PASS


In [5]:
rag_question = 'What safe initial response should an analyst take for a phishing message?'
expected_phrase = 'isolate the message, preserve full headers, and avoid opening links or attachments'
rag_hits = retriever.invoke(rag_question)
print('Retrieved chunks:', len(rag_hits))
for number, hit in enumerate(rag_hits, 1):
    print(f'[{number}] {hit.metadata.get("filename", hit.metadata.get("source"))}')
    print(hit.page_content[:500], '\n')
retrieved_text = '\n'.join(hit.page_content for hit in rag_hits).lower()
assert expected_phrase in retrieved_text
print('Verbatim retrieval assertion: PASS')

Retrieved chunks: 4
[1] phishing.md
# Phishing Indicators

Phishing assessment should combine message content, sender identity, link targets, attachments, and authentication evidence. Common indicators include urgent requests, suspicious sender domains, mismatched links, credential requests, unexpected attachments, and fear or urgency language. No single wording cue proves malicious intent.

Preserve the original message and full headers before taking action. A safe initial response is to isolate the message, preserve full headers 

[2] phishing.md
Useful header evidence includes `From`, `Reply-To`, `Return-Path`, received hops, and SPF, DKIM, and DMARC results. A display name that imitates a trusted organization while the underlying address uses an unrelated domain is a strong impersonation signal. Authentication failures matter, but legitimate forwarding can complicate interpretation.

Analysts should document which indicators were directly observed, which were returned by reputation

## 5. Context, short-term state, and long-term memory

Short-term workflow state uses an `InMemorySaver` and explicit `thread_id`. Long-term facts use a separate `InMemoryStore`. The test below writes in thread A and reads in thread B under the same analyst namespace, which proves that the fact is not merely checkpoint state or accumulated chat history.

In [6]:
from sentinel.memory.store import cross_thread_memory_workflow

memory_analyst = 'capstone-demo-analyst'
thread_a_config = {'configurable': {'thread_id': 'memory-thread-A'}}
thread_b_config = {'configurable': {'thread_id': 'memory-thread-B'}}
memory_write = cross_thread_memory_workflow.invoke({
    'operation': 'write', 'thread_label': 'thread-A', 'analyst_id': memory_analyst,
    'key': 'report_format', 'value': 'PDF'
}, thread_a_config)
memory_read = cross_thread_memory_workflow.invoke({
    'operation': 'read', 'thread_label': 'thread-B', 'analyst_id': memory_analyst,
    'key': 'report_format'
}, thread_b_config)
print('Write result:', memory_write)
print('Read result:', memory_read)
assert thread_a_config['configurable']['thread_id'] != thread_b_config['configurable']['thread_id']
assert memory_read['value'] == 'PDF'
print('Cross-thread Store assertion: PASS')

Write result: {'operation': 'write', 'thread_label': 'thread-A', 'analyst_id': 'capstone-demo-analyst', 'key': 'report_format', 'value': 'PDF'}
Read result: {'operation': 'read', 'thread_label': 'thread-B', 'analyst_id': 'capstone-demo-analyst', 'key': 'report_format', 'value': 'PDF'}
Cross-thread Store assertion: PASS


## 6. Human-in-the-loop — interrupt and resume

The full Sentinel workflow runs below. `force_human_review=True` is an explicit analyst policy override that guarantees a deterministic pause before the approved PDF is persisted. The first cell captures the interrupt and checkpoint; the second resumes the same thread with a structured reviewer decision.

In [7]:
from sentinel.workflows.workflow import sentinel_workflow

demo_started_at = datetime.now(timezone.utc)
hitl_config = {'configurable': {'thread_id': 'sentinel-hitl-capstone-demo'}}
workflow_input = {
    'request': 'Review this complete raw email for phishing indicators.\n\n' + raw_email,
    'analyst_id': memory_analyst,
    'force_human_review': True,
    'output_path': str(PROJECT_ROOT / 'reports' / 'approved_incident.pdf'),
}
first_run = sentinel_workflow.invoke(workflow_input, hitl_config)
print('First invocation:', first_run)
assert '__interrupt__' in first_run
interrupt_payload = first_run['__interrupt__'][0].value
assert interrupt_payload['action'] == 'persist_final_incident_report'
checkpoint = sentinel_workflow.get_state(hitl_config)
print('Saved checkpoint next tasks:', checkpoint.next)
print('Interrupt captured: PASS')

First invocation: {'__interrupt__': [Interrupt(value={'action': 'persist_final_incident_report', 'question': 'Approve persisting this incident report?', 'analysis': {'verdict': 'malicious', 'confidence': 0.85, 'threat_type': 'phishing', 'explanation': 'The email contains multiple indicators of phishing, including an urgent request for account verification, a generic sender address, and a suspicious reply-to address. The URL provided cannot be verified without further investigation, raising concerns about its legitimacy.', 'evidence': ['Urgent subject line', 'Request for account verification', 'Generic sender address', 'Suspicious reply-to address', 'No attachments present for analysis'], 'recommendations': ['Do not click on the link provided in the email.', 'Isolate the email and preserve full headers for further investigation.', 'Report the email as phishing to the appropriate security team.'], 'requires_human_approval': True}, 'required_response': {'approved': 'boolean', 'reviewer': 

In [8]:
from langgraph.types import Command

review_decision = {
    'approved': True,
    'reviewer': 'Capstone human reviewer',
    'reason': 'The evidence and limitations were reviewed before report persistence.',
}
resumed_run = sentinel_workflow.invoke(Command(resume=review_decision), hitl_config)
print(json.dumps(resumed_run, indent=2, default=str))
assert resumed_run['report']['approved'] is True
assert resumed_run['executed_tool_count'] >= 1
assert resumed_run['retrieved_documents'] >= 1
assert resumed_run['finalization']['status'] == 'written'
assert Path(resumed_run['finalization']['output_path']).exists()
print('Resume completed: PASS')

{
  "route": {
    "destination": "email_agent",
    "reason": "The investigation involves reviewing a complete raw email for phishing indicators, which falls under the domain of the email_agent."
  },
  "specialist": {
    "destination": "email_agent",
    "assessment": {
      "summary": "The email exhibits multiple indicators commonly associated with phishing attempts, including urgency in the subject line and a request for account verification. The sender's address appears generic, and the reply-to address is suspicious, which raises further concerns about the email's legitimacy.",
      "notable_indicators": [
        "Urgent subject line",
        "Request for account verification",
        "Generic sender address",
        "Suspicious reply-to address"
      ],
      "limitations": [
        "The analysis is based solely on the provided email content and does not include external verification of the URLs or sender authenticity.",
        "The tool used may not capture all phishi

## 7. Functional API and two error strategies

Sentinel uses LangGraph `@task` and `@entrypoint`. Model-dependent production tasks have a real `RetryPolicy`, while evidence-stage exceptions use a controlled `unknown` fallback. This credential-free demonstration makes both strategies visible: the transient dependency succeeds on the third framework-managed attempt, and the permanent failure reaches fallback without a handwritten sleep loop.

In [9]:
from sentinel.workflows.reliability_demo import reliability_demo
from sentinel.workflows.workflow import external_retry

reliability_result = reliability_demo.invoke({'run_id': str(uuid4())})
print('Production RetryPolicy:', external_retry)
print('Reliability demonstration:', reliability_result)
assert reliability_result['retry_result']['attempts'] == 3
assert reliability_result['retry_result']['status'] == 'recovered'
assert reliability_result['fallback_result']['strategy'] == 'controlled_fallback'
assert reliability_result['fallback_result']['verdict'] == 'unknown'
print('RetryPolicy assertion: PASS')
print('Controlled fallback assertion: PASS')

Production RetryPolicy: RetryPolicy(initial_interval=0.2, backoff_factor=2.0, max_interval=2.0, max_attempts=3, jitter=False, retry_on=<function default_retry_on at 0x119db65c0>)
Reliability demonstration: {'retry_policy': 'RetryPolicy', 'retry_result': {'status': 'recovered', 'attempts': 3}, 'fallback_result': {'verdict': 'unknown', 'confidence': 0.0, 'strategy': 'controlled_fallback', 'handled_error': 'ValueError'}}
RetryPolicy assertion: PASS
Controlled fallback assertion: PASS


## 8. Workflow pattern — Routing

**Declared workflow pattern: Routing.** It fits because email, URL, IP, and file artifacts are mutually exclusive dominant investigation modes with different tool permissions. An LLM supervisor makes the choice with structured output; shared retrieval, synthesis, approval, reporting, and memory stages then enforce consistent behavior.

In [10]:
print('Workflow Pattern: Routing')
print('Routing authority: structured LLM supervisor')
print('Why it fits: mutually exclusive artifact specialists with narrow tools')
assert resumed_run['route']['destination'] == 'email_agent'
print('End-to-end routing assertion: PASS')

Workflow Pattern: Routing
Routing authority: structured LLM supervisor
Why it fits: mutually exclusive artifact specialists with narrow tools
End-to-end routing assertion: PASS


## 9. LangSmith — verified trace and result-derived observation

The next cell waits for background trace upload, queries the actual configured project, finds a workflow root run created after this demonstration began, and derives an observation from its timings and child runs. A configured environment variable alone is not accepted as evidence.

In [11]:
from langchain_core.tracers.langchain import wait_for_all_tracers
from langsmith import Client

wait_for_all_tracers()
project_name = str(status['project'])
client = Client()
recent_runs = list(client.list_runs(
    project_name=project_name, start_time=demo_started_at - timedelta(seconds=5), limit=100
))
workflow_runs = [run for run in recent_runs if run.name == 'sentinel_workflow']
assert workflow_runs, f'No Sentinel workflow run found in LangSmith project {project_name}'
# Functional API traces use a LangGraph root with sentinel_workflow as a child.
# The earliest workflow run is the full pre-interrupt investigation; resume is a later trace.
workflow_run = min(workflow_runs, key=lambda run: run.start_time)
trace_runs = list(client.list_runs(project_name=project_name, trace_id=workflow_run.trace_id))
trace_root = next(run for run in trace_runs if run.parent_run_id is None)
completed_runs = [run for run in trace_runs if run.end_time is not None]
assert completed_runs, 'The trace has no completed runs'
duration = lambda run: (run.end_time - run.start_time).total_seconds()
slowest = max(completed_runs, key=duration)
error_count = sum(bool(run.error) for run in trace_runs)
tool_run_count = sum(run.run_type == 'tool' for run in trace_runs)
trace_observation = (
    f'Actual trace observation: trace {trace_root.trace_id} contained {len(trace_runs)} runs; '
    f'the slowest run was {slowest.name} at {duration(slowest):.3f}s; '
    f'{error_count} run(s) recorded errors; and the trace recorded '
    f'{tool_run_count} model-selected tool run(s).'
)
print('LangSmith project:', project_name)
print('Verified root trace ID:', trace_root.trace_id)
print(trace_observation)
assert len(trace_runs) > 1 and tool_run_count >= 1
print('LangSmith trace verification: PASS')

/var/folders/0_/lv9d6z550r9b696hy6scr69r0000gn/T/ipykernel_42085/3023798147.py:7: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  recent_runs = list(client.list_runs(


/var/folders/0_/lv9d6z550r9b696hy6scr69r0000gn/T/ipykernel_42085/3023798147.py:15: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  trace_runs = list(client.list_runs(project_name=project_name, trace_id=workflow_run.trace_id))


LangSmith project: sentinel-ai-capstone
Verified root trace ID: 01a03c09-d89c-7f20-ba0e-4516761a6458
Actual trace observation: trace 01a03c09-d89c-7f20-ba0e-4516761a6458 contained 18 runs; the slowest run was specialist_investigation at 5.326s; 0 run(s) recorded errors; and the trace recorded 1 model-selected tool run(s).
LangSmith trace verification: PASS


## 10. Automated regression suite

The suite validates tool execution, schemas, local analysis, structured routing, deterministic RAG retrieval, cross-thread memory, HITL resume, both error strategies, and PDF generation without spending API credits.

In [12]:
import subprocess

test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=PROJECT_ROOT, text=True, capture_output=True, check=False
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
assert test_run.returncode == 0
print('Automated regression suite: PASS')

..............                                                           [100%]
14 passed in 1.79s

Automated regression suite: PASS


## 11. Final evidence summary

The executed results above—not the implementation plan—show a model-selected tool call, four structured routes, a verbatim RAG hit, Store memory crossing thread IDs, a saved HITL checkpoint, successful resume and PDF persistence, RetryPolicy recovery, controlled fallback, the explicitly named Routing pattern, a real LangSmith trace observation, and a passing regression suite.

In [13]:
final_evidence = {
    'model_selected_tools': len(specialist_demo.tool_evidence),
    'routing_destinations': sorted(decision.destination for decision in route_results),
    'rag_chunks': len(chunks),
    'cross_thread_memory': memory_read['value'],
    'hitl_resumed_and_approved': resumed_run['report']['approved'],
    'pdf_written': resumed_run['finalization']['status'],
    'retry_attempts': reliability_result['retry_result']['attempts'],
    'workflow_pattern': 'Routing',
    'langsmith_trace_id': str(trace_root.trace_id),
    'tests_passed': test_run.returncode == 0,
}
print(json.dumps(final_evidence, indent=2))
assert all([
    final_evidence['model_selected_tools'] >= 1,
    final_evidence['cross_thread_memory'] == 'PDF',
    final_evidence['hitl_resumed_and_approved'],
    final_evidence['pdf_written'] == 'written',
    final_evidence['retry_attempts'] == 3,
    final_evidence['tests_passed'],
])
print('FINAL CAPSTONE VERIFICATION: PASS')

{
  "model_selected_tools": 1,
  "routing_destinations": [
    "email_agent",
    "file_agent",
    "ip_agent",
    "url_agent"
  ],
  "rag_chunks": 10,
  "cross_thread_memory": "PDF",
  "hitl_resumed_and_approved": true,
  "pdf_written": "written",
  "retry_attempts": 3,
  "workflow_pattern": "Routing",
  "langsmith_trace_id": "01a03c09-d89c-7f20-ba0e-4516761a6458",
  "tests_passed": true
}
FINAL CAPSTONE VERIFICATION: PASS
